# Legal LLM Fine-Tuning (QLoRA)

**One-click pipeline:** Trains a LoRA adapter on Saul-7B using your prepared dataset.

**Prerequisites:**
1. Run steps 1-5 on your laptop to build `data/training/`
2. Upload `data/training/` folder to Google Drive at `My Drive/legal-finetune/data/training/`
3. Open this notebook in Colab with **A100 GPU** runtime
4. Run all cells

**Output:** Merged model saved to `My Drive/legal-finetune/outputs/merged/` — ready for vLLM deployment.

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0
print(f'\nGPU: {gpu_name} ({vram_gb:.0f} GB)')
assert torch.cuda.is_available(), 'No GPU! Change runtime: Runtime → Change runtime type → A100'
assert vram_gb >= 15, f'Need >= 16GB VRAM, got {vram_gb:.0f}GB. Switch to A100 or T4.'

## Cell 2 — Install dependencies

In [ ]:
!pip install -q torch transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 \
    accelerate>=0.30.0 datasets>=2.19.0 trl>=0.8.0 rouge-score jsonlines pyyaml

## Cell 3 — Mount Google Drive & setup paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# All paths on Google Drive
DRIVE_BASE = Path('/content/drive/MyDrive/legal-finetune')
DATASET_DIR = DRIVE_BASE / 'data' / 'training'
OUTPUT_DIR = DRIVE_BASE / 'outputs' / 'saul-7b-legal-lora'
MERGED_DIR = DRIVE_BASE / 'outputs' / 'merged'

# Verify dataset exists
assert DATASET_DIR.exists(), (
    f'Dataset not found at {DATASET_DIR}\n'
    f'Upload data/training/ from your laptop to Google Drive at:\n'
    f'  My Drive/legal-finetune/data/training/'
)

# Check what's in the dataset
for f in sorted(DATASET_DIR.rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.relative_to(DATASET_DIR)} ({size_mb:.1f} MB)')

print(f'\nDataset: {DATASET_DIR}')
print(f'Output:  {OUTPUT_DIR}')
print(f'Merged:  {MERGED_DIR}')

## Cell 4 — HuggingFace login (for gated models)

In [ ]:
# Only needed if Equall/Saul-Instruct-v1 is gated
# Get token from: https://huggingface.co/settings/tokens
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'  # <-- replace this

if HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 10:
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF token set')
else:
    print('No HF token — will work if model is public')

## Cell 5 — Load dataset & show stats

In [ ]:
from datasets import DatasetDict
from collections import Counter

dataset = DatasetDict.load_from_disk(str(DATASET_DIR))

print(f'Train:      {len(dataset["train"]):,} examples')
print(f'Validation: {len(dataset["validation"]):,} examples')
print()

# Distribution
for split in ['train', 'validation']:
    print(f'--- {split} ---')
    types = Counter(dataset[split]['clause_type'])
    sources = Counter(dataset[split]['source'])
    for ct, n in types.most_common():
        print(f'  {ct}: {n}')
    print(f'  Sources: {dict(sources)}')
    print()

# Preview one example
print('=== Sample ====')
print(dataset['train'][0]['text'][:500])

## Cell 6 — Configure training

In [ ]:
import torch

# ── Model ──
BASE_MODEL = 'Equall/Saul-Instruct-v1'
MAX_SEQ_LENGTH = 2048

# ── QLoRA ──
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                   'gate_proj', 'up_proj', 'down_proj']

# ── Training ──
NUM_EPOCHS = 3
BATCH_SIZE = 4        # per device
GRAD_ACCUM = 4        # effective batch = 16
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05

# ── Auto-detect precision ──
USE_BF16 = torch.cuda.is_bf16_supported()
print(f'BF16 supported: {USE_BF16}')

# ── Estimate training time ──
total_steps = (len(dataset['train']) * NUM_EPOCHS) // (BATCH_SIZE * GRAD_ACCUM)
print(f'Estimated steps: {total_steps}')
print(f'Estimated time: ~{total_steps * 3 / 60:.0f} min on A100')

## Cell 7 — Load model in 4-bit + attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 8 — Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    bf16=USE_BF16,
    fp16=not USE_BF16,
    logging_steps=10,
    save_steps=100,
    eval_strategy='steps',
    eval_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to='none',
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f'Starting training: {NUM_EPOCHS} epochs, {len(dataset["train"]):,} examples...')
trainer.train()
print('Training complete!')

## Cell 9 — Save adapter to Drive

In [ ]:
adapter_path = OUTPUT_DIR / 'final_adapter'
adapter_path.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

# Check saved size
adapter_size = sum(f.stat().st_size for f in adapter_path.rglob('*') if f.is_file()) / 1e6
print(f'Adapter saved to: {adapter_path}')
print(f'Adapter size: {adapter_size:.0f} MB')

## Cell 10 — Quick evaluation

In [ ]:
import re

# Test prompts — one per clause type
TEST_PROMPTS = [
    'Draft a Payment Terms clause for a SaaS Subscription Agreement governed by California, United States law.',
    'Draft a Termination clause for a Master Services Agreement governed by New York, United States law.',
    'Draft a Confidentiality clause for a Non-Disclosure Agreement governed by Delaware, United States law.',
    'Draft a Force Majeure clause for a Supply Agreement governed by England and Wales law.',
    'Draft an Intellectual Property Rights clause for a Software License Agreement governed by India law.',
]

model.eval()
print('=== QUICK EVALUATION ===')
print()

for prompt in TEST_PROMPTS:
    input_text = f'<s>[INST] {prompt} [/INST]'
    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    generated = tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    
    # Quick quality checks
    has_placeholders = bool(re.search(r'\[[^\]]{2,}\]|\bTBD\b|\bTBC\b', generated))
    sub_clause_count = len(re.findall(r'^\d+[.)\s]', generated, re.MULTILINE))
    word_count = len(generated.split())
    
    clause_type = prompt.split('clause')[0].split('a ')[-1].strip()
    status = 'PASS' if (not has_placeholders and sub_clause_count >= 3 and word_count >= 100) else 'FAIL'
    
    print(f'[{status}] {clause_type}')
    print(f'  Sub-clauses: {sub_clause_count} | Words: {word_count} | Placeholders: {has_placeholders}')
    print(f'  Preview: {generated[:200]}...')
    print()

## Cell 11 — Merge adapter into base model (for vLLM deployment)

In [ ]:
# Free GPU memory from training
del model
del trainer
torch.cuda.empty_cache()
import gc
gc.collect()

print('Loading base model in fp16 for merging...')
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print('Loading adapter...')
merged_model = PeftModel.from_pretrained(base_model, str(adapter_path))

print('Merging weights...')
merged_model = merged_model.merge_and_unload()

print(f'Saving merged model to: {MERGED_DIR}')
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))

merged_size = sum(f.stat().st_size for f in MERGED_DIR.rglob('*') if f.is_file()) / 1e9
print(f'Merged model size: {merged_size:.1f} GB')
print()
print('DONE! Your fine-tuned model is at:')
print(f'  Google Drive: My Drive/legal-finetune/outputs/merged/')
print()
print('To deploy, download to your GCP VM and run:')
print(f'  vllm serve ./merged --dtype half --chat-template /tmp/mistral.jinja')

## Cell 12 — Compare base vs fine-tuned (optional)

In [ ]:
# Side-by-side comparison: base Saul vs fine-tuned
COMPARE_PROMPT = 'Draft a Payment Terms clause for a SaaS Subscription Agreement governed by California, United States law.'

input_text = f'<s>[INST] {COMPARE_PROMPT} [/INST]'
inputs = tokenizer(input_text, return_tensors='pt').to(merged_model.device)

# Fine-tuned output
with torch.no_grad():
    ft_output = merged_model.generate(**inputs, max_new_tokens=1024, temperature=0.1,
                                       do_sample=True, pad_token_id=tokenizer.pad_token_id)
ft_text = tokenizer.decode(ft_output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# Base model output
del merged_model
torch.cuda.empty_cache()
gc.collect()

base_model_fresh = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                           bnb_4bit_compute_dtype=torch.float16),
    device_map='auto', trust_remote_code=True,
)
inputs = tokenizer(input_text, return_tensors='pt').to(base_model_fresh.device)
with torch.no_grad():
    base_output = base_model_fresh.generate(**inputs, max_new_tokens=1024, temperature=0.1,
                                             do_sample=True, pad_token_id=tokenizer.pad_token_id)
base_text = tokenizer.decode(base_output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('=== BASE MODEL ===')
print(base_text[:800])
print()
print('=== FINE-TUNED ===')
print(ft_text[:800])